# Day 25 Tutorial：集成分歧诊断

> **课程附带教程。** 预测标准差是启发式模型分歧，不是 95% 预测区间，也不是下游任务实验误差。

## Goal

只在 ESOL 原 train 内建立 scaffold 隔离诊断 holdout，训练五个只改变随机种子的随机森林，并诊断集成分歧；外部 valid 不参与本日规则设计。

## Setup

固定成员参数与种子列表。内部诊断标签只在分歧计算完成后使用；外部 valid 标签不读取，留给 Day 26 的固定同轮策略对照。

In [1]:
from pathlib import Path
import contextlib
import io
import warnings

from rdkit import RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-Learning 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, _external_valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)

fit_idx, diagnostic_idx, unused_idx = dc.splits.ScaffoldSplitter().split(
    train_dataset,
    frac_train=0.8,
    frac_valid=0.2,
    frac_test=0.0,
)
fit_idx = np.asarray(fit_idx)
diagnostic_idx = np.asarray(diagnostic_idx)
assert len(unused_idx) == 0

# 独立复核标准 Murcko scaffold；空字符串也保持为同一个组。
scaffold_groups = np.asarray([
    MurckoScaffold.MurckoScaffoldSmiles(
        smiles=smiles, includeChirality=False
    )
    for smiles in train_ids
])
X_fit, y_fit = X_train[fit_idx], y_train[fit_idx]
X_diagnostic = X_train[diagnostic_idx]
y_diagnostic = y_train[diagnostic_idx]
diagnostic_ids = train_ids[diagnostic_idx]
print("Task:", tasks[0])
print("Internal fit / diagnostic:", X_fit.shape, X_diagnostic.shape)
print("External valid labels accessed: no")
print("测试集对象保持封存，本教程不创建测试预测。")

Task: measured log solubility in mols per litre
Internal fit / diagnostic: (721, 1024) (181, 1024)
External valid labels accessed: no
测试集对象保持封存，本教程不创建测试预测。


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

MEMBER_SEEDS = [11, 22, 33, 44, 55]

def make_member(seed):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=120,
            max_features="sqrt",
            random_state=seed,
            n_jobs=1,
        ),
    )

## Steps

### 1. 生成“模型 × 样本”预测矩阵

In [3]:
member_predictions = []
for seed in MEMBER_SEEDS:
    member = make_member(seed)
    member.fit(X_fit, y_fit)
    member_predictions.append(member.predict(X_diagnostic))

prediction_matrix = np.vstack(member_predictions)
mean_prediction = prediction_matrix.mean(axis=0)
disagreement = prediction_matrix.std(axis=0, ddof=0)
print("Prediction matrix:", prediction_matrix.shape)

Prediction matrix: (5, 181)


### 2. 内部 holdout 标签只用于事后诊断

候选池阶段不能创建 `absolute_error`，因为候选真实标签未知。这里使用的标签只来自预先切出的内部诊断 holdout。

In [4]:
diagnosis = pd.DataFrame({
    "sample_id": diagnostic_ids,
    "prediction_mean": mean_prediction,
    "disagreement": disagreement,
})
# 诊断边界：下面一列只在原 train 的内部 holdout 上计算。
diagnosis["absolute_error"] = np.abs(y_diagnostic - mean_prediction)
diagnosis = diagnosis.sort_values(
    ["disagreement", "sample_id"],
    ascending=[False, True],
).reset_index(drop=True)
spearman_rho, spearman_p = spearmanr(
    diagnosis["disagreement"],
    diagnosis["absolute_error"],
)
print({"spearman_rho": spearman_rho, "spearman_p": spearman_p})
display(diagnosis.head(10).round(4))

{'spearman_rho': np.float64(-0.1827343108948137), 'spearman_p': np.float64(0.01380976717519863)}


,sample_id,prediction_mean,disagreement,absolute_error
0,O=C3CN=C(c1ccccc1)c2cc(ccc2N3)N(=O)=O,-3.7030,0.3391,0.0930
1,Sc1nccc(=O)[nH]1,-1.0995,0.3102,1.1735
2,CN2C(=O)CN=C(c1ccccc1)c3cc(ccc23)N(=O)=O,-3.7281,0.2805,0.0679
3,c1ccc2ncccc2c1,-1.1552,0.2735,0.1448
4,NC(=O)NC1NC(=O)NC1=O,-1.7085,0.2681,0.1085
5,CN(C)C(=O)Oc1nc(nc(C)c1C)N(C)C,-2.0447,0.2669,0.0947
6,CC(C)C1CCC(C)CC1O,-2.4138,0.2594,0.1162
7,O=C1N(COC(=O)CCCCC)C(=O)C(N1)(c2ccccc2)c3ccccc3,-3.0177,0.2563,2.8683
8,c3ccc2nc1ccccc1cc2c3,-3.4240,0.2550,0.2460
9,CCOP(=S)(OCC)SC(CCl)N1C(=O)c2ccccc2C1=O,-4.3033,0.2477,2.0367


### 3. 比较低/中/高分歧组

分组结果可能不呈单调关系；负结果也必须保留。

In [5]:
rank_fraction = diagnosis["disagreement"].rank(method="first", pct=True)
diagnosis["disagreement_group"] = pd.cut(
    rank_fraction,
    bins=[0.0, 1 / 3, 2 / 3, 1.0],
    labels=["low", "middle", "high"],
    include_lowest=True,
)
group_summary = diagnosis.groupby(
    "disagreement_group", observed=True
).agg(
    n=("sample_id", "size"),
    mean_disagreement=("disagreement", "mean"),
    mean_absolute_error=("absolute_error", "mean"),
)
display(group_summary.round(4))

,n,mean_disagreement,mean_absolute_error
disagreement_group,,,
low,60,0.0716,1.7151
middle,60,0.1357,1.6183
high,61,0.2061,1.3039


## Checks

轴方向、成员数、有限数值和样本 ID 唯一性必须通过。

In [6]:
assert set(scaffold_groups[fit_idx]).isdisjoint(
    set(scaffold_groups[diagnostic_idx])
)
assert prediction_matrix.shape == (len(MEMBER_SEEDS), len(y_diagnostic))
assert mean_prediction.shape == disagreement.shape == (len(y_diagnostic),)
assert np.isfinite(prediction_matrix).all()
assert np.isfinite([spearman_rho, spearman_p]).all()
assert (disagreement >= 0).all()
assert diagnosis["sample_id"].is_unique
print("Checks passed. External valid labels were not accessed.")
print("This disagreement has no calibrated coverage claim.")

Checks passed. External valid labels were not accessed.
This disagreement has no calibrated coverage claim.


## Next Steps

冻结成员、`ddof=0`、排序规则和预算后进入 Day 26 池模拟，并始终保留同预算随机对照。真实候选还必须通过化学可行性与安全审核。